# FinDER: Vector RAG vs Graph RAG (LanceDB + Neo4j)

This tutorial builds two parallel retrieval pipelines over the **FinDER** SEC 10-K Q&A dataset and answers the same questions through each:

1. **Vector RAG** — embed chunks, top-k similarity, augment the prompt. Backed by `LanceDBVectorStore`.
2. **Graph RAG** — entity extraction + linking, retrieve neighbors via Cypher, augment the prompt. Backed by `Neo4jGraphStore` against the bundled DozerDB instance (apoc + n10s).
3. **Hybrid** — pass both vector hits and graph context into the answer synthesizer.

**Why Neo4j for graph?** Upstream [lance-graph](https://github.com/lance-format/lance-graph/issues/91) is in development and not yet usable as a property-graph backend. For the demo we use the DozerDB instance that ships with the tutorial Docker compose. The `examples/finder/lib/lance_graph_store.py` adapter stays in the repo as a forward-compatible reference for when lance-graph stabilizes.

**Visualization.** The last section draws the constructed graph inline via NetworkX/matplotlib and points you at the **Neo4j Browser** (http://localhost:7474) for live exploration.

**Dataset.** Set `FINDER_PATH` env var to your FinDER JSON. If unset, this notebook uses `examples/finder/datasets/finder_tutorial_subset.json` so the cells run end-to-end without external data.

## Setup

In [3]:
import json
import os
import sys
import time
from pathlib import Path

from dotenv import load_dotenv

def _find_repo_root() -> Path:
    p = Path.cwd().resolve()
    while p != p.parent:
        if (p / "seocho").is_dir() and (p / "examples").is_dir():
            return p
        p = p.parent
    return Path.cwd()

ROOT = _find_repo_root()
sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env")

LLM_SPEC = os.environ.get("SEOCHO_LLM", "openai/gpt-4o-mini")
LLM_PROVIDER, LLM_MODEL = (LLM_SPEC.split("/", 1) + [""])[:2]
if not LLM_MODEL:
    LLM_PROVIDER, LLM_MODEL = "openai", LLM_SPEC

PROVIDER_KEY = {
    "openai": "OPENAI_API_KEY",
    "deepseek": "DEEPSEEK_API_KEY",
    "kimi": "MOONSHOT_API_KEY",
    "grok": "XAI_API_KEY",
    "qwen": "DASHSCOPE_API_KEY",
}.get(LLM_PROVIDER, "OPENAI_API_KEY")

if not os.environ.get(PROVIDER_KEY):
    raise RuntimeError(f"{PROVIDER_KEY} required for SEOCHO_LLM={LLM_SPEC}")
if not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError(
        "OPENAI_API_KEY is required for Tutorial 1's vector path "
        "(embeddings must use OpenAI even when SEOCHO_LLM points elsewhere)."
    )

NEO4J_URI      = os.environ.get("NEO4J_URI", "bolt://tutorials-neo4j:7687")
NEO4J_USER     = os.environ.get("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.environ.get("NEO4J_PASSWORD", "tutorialspw")

FINDER_PATH = os.environ.get(
    "FINDER_PATH",
    str(ROOT / "examples" / "finder" / "datasets" / "finder_tutorial_subset.json"),
)
WORKDIR = ROOT / ".seocho" / "finder_tutorial"
WORKDIR.mkdir(parents=True, exist_ok=True)
WORKSPACE_ID = "finder_tutorial"

print(f"FinDER source:  {FINDER_PATH}")
print(f"Working dir:    {WORKDIR}")
print(f"LLM:            {LLM_PROVIDER}/{LLM_MODEL}")
print(f"Embeddings:     openai/text-embedding-3-small (fixed)")
print(f"Graph backend:  Neo4j @ {NEO4J_URI}")

FinDER source:  /workspace/examples/datasets/finder_tutorial_subset.json
Working dir:    /workspace/.seocho/finder_tutorial
LLM:            openai/gpt-4o-mini
Embeddings:     openai/text-embedding-3-small (fixed)
Graph backend:  Neo4j @ bolt://tutorials-neo4j:7687


In [ ]:
from seocho.benchmarking import load_finder_cases

cases = load_finder_cases(FINDER_PATH)
print(f"Loaded {len(cases)} FinDER cases")
for case in cases[:3]:
    print(f"  [{case.category}] {case.case_id}: {case.question}")

## Vector RAG path

In [18]:
from seocho.store.vector import create_vector_store

vector_store = create_vector_store(
    kind="lancedb",
    uri=str(WORKDIR / "vec.lance"),
    table_name="finder_vec",
)

items = [
    {
        "id": case.case_id,
        "text": case.text,
        "metadata": {"category": case.category, "question": case.question},
    }
    for case in cases
]
added = vector_store.add_batch(items)
print(f"Indexed {added} chunks into LanceDB. Active rows: {vector_store.count()}")

Indexed 10 chunks into LanceDB. Active rows: 10


In [19]:
from seocho.store.llm import create_llm_backend

llm = create_llm_backend(provider=LLM_PROVIDER, model=LLM_MODEL)

ANSWER_SYSTEM = "Answer the question using only the supplied evidence. Be concise. If the evidence does not contain the answer, say so."

def answer_with_vector(question: str, *, k: int = 3) -> dict:
    t0 = time.perf_counter()
    hits = vector_store.search(question, limit=k)
    retrieval_ms = (time.perf_counter() - t0) * 1000
    context = "\n\n".join(f"[{h.id}] {h.text}" for h in hits)
    user = f"Context:\n{context}\n\nQuestion: {question}"
    t1 = time.perf_counter()
    response = llm.complete(system=ANSWER_SYSTEM, user=user)
    gen_ms = (time.perf_counter() - t1) * 1000
    return {
        "answer": response.text.strip(),
        "retrieval_ms": retrieval_ms,
        "generation_ms": gen_ms,
        "context_chars": len(context),
        "hit_count": len(hits),
    }

vec_demo = answer_with_vector(cases[0].question)
print(f"Q: {cases[0].question}")
print(f"A: {vec_demo['answer']}")
print(f"   ({vec_demo['retrieval_ms']:.1f} ms retrieve, {vec_demo['generation_ms']:.1f} ms generate, {vec_demo['hit_count']} hits)")

Q: Where is Apple Inc. headquartered?
A: Apple Inc. is headquartered in Cupertino, California.
   (2197.5 ms retrieve, 1367.3 ms generate, 3 hits)


## Graph RAG path (Neo4j)

Tiny ontology (Company / Person / FinancialMetric / Regulation, from `examples/datasets/fibo_base.jsonld`). Extraction runs through `IndexingPipeline`, writes go through seocho's standard `Neo4jGraphStore` over Bolt, and retrieval is plain Cypher — same backend and contract as production.

In [ ]:
from seocho import Ontology, Seocho
from seocho.store.graph import Neo4jGraphStore

ontology = Ontology.from_jsonld(ROOT / "examples" / "datasets" / "fibo_base.jsonld")
graph_store = Neo4jGraphStore(NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD)

# Wipe any leftover nodes from a previous run of this tutorial.
graph_store.execute_write(
    "MATCH (n) WHERE n._workspace_id = $workspace_id DETACH DELETE n",
    params={"workspace_id": WORKSPACE_ID},
)
graph_store.ensure_constraints(ontology)

client = Seocho(
    ontology=ontology,
    graph_store=graph_store,
    llm=llm,
    workspace_id=WORKSPACE_ID,
)
# Pin to the default Neo4j database. By default Seocho derives a per-ontology
# database name (e.g. 'fibobaselpg') and tries to write there; on this single-
# database DozerDB instance only 'neo4j' exists, so writes would fail with
# DatabaseNotFound. Workspaces inside the 'neo4j' database give us the same
# logical separation without needing a CREATE DATABASE step.
client.default_database = "neo4j"

for case in cases:
    client.add(case.text, metadata={"case_id": case.case_id, "category": case.category})

schema = graph_store.get_schema()
print(f"Labels:         {schema.get('labels', [])}")
print(f"Rel types:      {schema.get('relationship_types', [])}")
counts = graph_store.query(
    "MATCH (n) WHERE n._workspace_id = $workspace_id "
    "RETURN count(n) AS nodes",
    params={"workspace_id": WORKSPACE_ID},
)
edge_counts = graph_store.query(
    "MATCH ()-[r]->() WHERE r._workspace_id = $workspace_id "
    "RETURN count(r) AS rels",
    params={"workspace_id": WORKSPACE_ID},
)
print(f"Workspace size: {counts[0]['nodes']} nodes, {edge_counts[0]['rels']} rels")

In [21]:
# Cypher-driven graph retrieval: pull seed nodes whose name matches a
# question keyword, then expand one hop. Returns a flat list of facts.
import re

STOPWORDS = {"what", "where", "who", "when", "how", "the", "a", "an", "is", "was", "were", "of", "in", "to", "for", "on", "and", "did", "during"}
TOKEN_RE = re.compile(r"[A-Za-z][A-Za-z0-9_.&\-]+")

def graph_context(question: str, *, seed_limit: int = 3, hop_limit: int = 5) -> str:
    tokens = [t for t in TOKEN_RE.findall(question) if t.lower() not in STOPWORDS]
    facts: list[str] = []
    seen: set[str] = set()
    for tok in tokens:
        seeds = graph_store.query(
            "MATCH (n) "
            "WHERE n._workspace_id = $workspace_id "
            "  AND toLower(n.name) CONTAINS toLower($kw) "
            "RETURN n.id AS id, labels(n)[0] AS label, n.name AS name, properties(n) AS props "
            "LIMIT $seed_limit",
            params={"workspace_id": WORKSPACE_ID, "kw": tok, "seed_limit": seed_limit},
        )
        for seed in seeds:
            if seed["id"] in seen:
                continue
            seen.add(seed["id"])
            facts.append(f"{seed['label']}({seed['name']}) properties={seed['props']}")
            hops = graph_store.query(
                "MATCH (n {id: $id, _workspace_id: $workspace_id})-[r]-(m) "
                "RETURN m.id AS neighbor_id, labels(m)[0] AS neighbor_label, "
                "       m.name AS neighbor_name, type(r) AS edge_type, "
                "       CASE WHEN startNode(r) = n THEN 'out' ELSE 'in' END AS direction "
                "LIMIT $hop_limit",
                params={"id": seed["id"], "workspace_id": WORKSPACE_ID, "hop_limit": hop_limit},
            )
            for hop in hops:
                arrow = "->" if hop["direction"] == "out" else "<-"
                facts.append(
                    f"{seed['name']} {arrow}[{hop['edge_type']}]{arrow} "
                    f"{hop['neighbor_label']}({hop['neighbor_name']})"
                )
    return "\n".join(facts)

demo_ctx = graph_context(cases[0].question)
print("Graph context for first question:")
print(demo_ctx or "(no matching nodes — try another question)")

Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `_workspace_id` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=19, offset=18>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 18, 'line': 1, 'column': 19}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'MATCH (n) WHERE n._workspace_id = $workspace_id   AND toLower(n.name) CONTAINS toLower($kw) RETURN n.id AS id, labels(n)[0] AS label, n.name AS name, properties(n) AS props LIMIT $seed_limit'
Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The p

Graph context for first question:
(no matching nodes — try another question)


In [22]:
def answer_with_graph(question: str) -> dict:
    t0 = time.perf_counter()
    context = graph_context(question)
    retrieval_ms = (time.perf_counter() - t0) * 1000
    if not context:
        return {
            "answer": "(no graph evidence)",
            "retrieval_ms": retrieval_ms,
            "generation_ms": 0.0,
            "context_chars": 0,
            "hit_count": 0,
        }
    user = f"Graph evidence:\n{context}\n\nQuestion: {question}"
    t1 = time.perf_counter()
    response = llm.complete(system=ANSWER_SYSTEM, user=user)
    gen_ms = (time.perf_counter() - t1) * 1000
    return {
        "answer": response.text.strip(),
        "retrieval_ms": retrieval_ms,
        "generation_ms": gen_ms,
        "context_chars": len(context),
        "hit_count": context.count("\n") + 1,
    }

graph_demo = answer_with_graph(cases[0].question)
print(f"Q: {cases[0].question}")
print(f"A: {graph_demo['answer']}")
print(f"   ({graph_demo['retrieval_ms']:.1f} ms retrieve, {graph_demo['generation_ms']:.1f} ms generate, {graph_demo['hit_count']} graph facts)")

Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `_workspace_id` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=19, offset=18>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 18, 'line': 1, 'column': 19}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'MATCH (n) WHERE n._workspace_id = $workspace_id   AND toLower(n.name) CONTAINS toLower($kw) RETURN n.id AS id, labels(n)[0] AS label, n.name AS name, properties(n) AS props LIMIT $seed_limit'
Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The p

Q: Where is Apple Inc. headquartered?
A: (no graph evidence)
   (19.6 ms retrieve, 0.0 ms generate, 0 graph facts)


## Graph RAG: text2Cypher variant

The cell above retrieves with a fixed pattern (keyword → seed nodes → 1-hop). That's deterministic but blunt — it can't follow multi-hop paths or aggregate. The other end of the spectrum is **text2Cypher**: hand the LLM the live schema, ask it to write Cypher for the question, execute, then synthesize the answer from the records.

Tradeoff in one line: text2Cypher is more expressive but trades a deterministic Cypher template for an LLM hallucination risk. Both retrieval styles stay in this notebook so you can compare.

The snippet below:

1. pulls the actual schema (labels, relationship types, sample property keys per label) from the live graph
2. prompts the LLM with the schema + the question, constrained to read-only Cypher and the active `$workspace_id`
3. strips code fences and refuses any Cypher that contains a write keyword (`CREATE`, `MERGE`, `SET`, `DELETE`, …)
4. executes the Cypher against Neo4j, then asks the LLM to summarize the records into a one-sentence answer
5. reports plan / exec / gen timings separately so you can see where the cost goes

In [ ]:
# --- text2Cypher: build a live schema, prompt, execute, synthesize ---
import json

# Sample one node per label to learn what property keys each label uses.
# This makes the LLM's Cypher reference real properties rather than guessing.
sample_props_by_label: dict[str, list[str]] = {}
for label in graph_store.get_schema().get("labels", []):
    rows = graph_store.query(
        f"MATCH (n:{label}) WHERE n._workspace_id = $workspace_id "
        "RETURN keys(n) AS k LIMIT 1",
        params={"workspace_id": WORKSPACE_ID},
    )
    if rows:
        sample_props_by_label[label] = sorted(
            k for k in rows[0]["k"] if not k.startswith("_")
        )

schema_block = (
    f"Node labels: {sorted(graph_store.get_schema().get('labels', []))}\n"
    f"Relationship types: {sorted(graph_store.get_schema().get('relationship_types', []))}\n"
    f"Properties per label:\n"
    + "\n".join(f"  {l}: {ps}" for l, ps in sample_props_by_label.items())
)

T2C_SYSTEM = (
    "You are a graph database engineer. Given a Neo4j schema and a "
    "natural-language question, write a single read-only Cypher query "
    "that answers it.\n"
    "Constraints:\n"
    "- Read-only operators only (no CREATE / MERGE / SET / DELETE / REMOVE)\n"
    "- Always filter to WHERE n._workspace_id = $workspace_id "
    "(and r._workspace_id = $workspace_id when matching relationships)\n"
    "- Use only the labels and relationship types listed in the schema\n"
    "- RETURN concrete property values (e.g. n.name, m.value), not whole nodes\n"
    "- Add LIMIT 20 if there is no aggregation\n"
    "- Output ONLY the Cypher query (no markdown fences, no commentary)"
)

WRITE_KEYWORDS = ("CREATE", "MERGE", "SET ", "DELETE", "REMOVE", "DROP", "DETACH ")

def cypher_from_question(question: str) -> str:
    """Use the LLM to translate a natural-language question into Cypher."""
    user = f"Schema:\n{schema_block}\n\nQuestion: {question}\n\nCypher:"
    cypher = llm.complete(system=T2C_SYSTEM, user=user).text.strip()
    # Strip ```cypher ... ``` fences if the LLM ignored the no-fence rule
    cypher = "\n".join(
        line for line in cypher.splitlines()
        if not line.strip().startswith("```")
    ).strip()
    upper = cypher.upper()
    if any(kw in upper for kw in WRITE_KEYWORDS):
        raise ValueError(f"refused: cypher contains write keyword:\n{cypher}")
    return cypher


def answer_with_text2cypher(question: str) -> dict:
    t0 = time.perf_counter()
    cypher = cypher_from_question(question)
    plan_ms = (time.perf_counter() - t0) * 1000

    t1 = time.perf_counter()
    try:
        records = graph_store.query(
            cypher, params={"workspace_id": WORKSPACE_ID},
        )
    except Exception as exc:
        return {
            "cypher": cypher, "records": [],
            "answer": f"<exec error: {type(exc).__name__}: {exc}>",
            "plan_ms": plan_ms,
            "exec_ms": (time.perf_counter() - t1) * 1000,
            "gen_ms": 0.0,
        }
    exec_ms = (time.perf_counter() - t1) * 1000

    if not records:
        return {
            "cypher": cypher, "records": [], "answer": "(no records)",
            "plan_ms": plan_ms, "exec_ms": exec_ms, "gen_ms": 0.0,
        }

    t2 = time.perf_counter()
    user = (
        f"Question: {question}\n\n"
        f"Cypher results (top 10 rows):\n"
        f"{json.dumps(records[:10], indent=2, default=str)}\n\n"
        "Answer concisely:"
    )
    answer = llm.complete(
        system="Answer in one sentence using only the supplied records.",
        user=user,
    ).text.strip()
    gen_ms = (time.perf_counter() - t2) * 1000

    return {
        "cypher": cypher, "records": records, "answer": answer,
        "plan_ms": plan_ms, "exec_ms": exec_ms, "gen_ms": gen_ms,
    }


# --- demo ---
demo = answer_with_text2cypher(cases[0].question)
print(f"Q: {cases[0].question}\n")
print("Generated Cypher:")
print(demo["cypher"])
print(f"\nRecords: {len(demo['records'])} rows")
for r in demo["records"][:3]:
    print(f"  {r}")
print(f"\nA: {demo['answer']}")
print(
    f"   (plan {demo['plan_ms']:.0f} ms · "
    f"exec {demo['exec_ms']:.0f} ms · gen {demo['gen_ms']:.0f} ms)"
)

## Hybrid: vector + graph context

In [ ]:
def answer_hybrid(question: str, *, k: int = 3) -> dict:
    t0 = time.perf_counter()
    hits = vector_store.search(question, limit=k)
    vec_ctx = "\n\n".join(f"[{h.id}] {h.text}" for h in hits)
    g_ctx = graph_context(question)
    retrieval_ms = (time.perf_counter() - t0) * 1000
    user = (
        f"Passages:\n{vec_ctx}\n\nGraph evidence:\n{g_ctx}\n\n"
        f"Question: {question}"
    )
    t1 = time.perf_counter()
    response = llm.complete(system=ANSWER_SYSTEM, user=user)
    gen_ms = (time.perf_counter() - t1) * 1000
    return {
        "answer": response.text.strip(),
        "retrieval_ms": retrieval_ms,
        "generation_ms": gen_ms,
        "context_chars": len(vec_ctx) + len(g_ctx),
        "hit_count": len(hits) + (g_ctx.count("\n") + 1 if g_ctx else 0),
    }

hyb_demo = answer_hybrid(cases[0].question)
print(f"Q: {cases[0].question}")
print(f"A: {hyb_demo['answer']}")

## Side-by-side comparison

In [ ]:
from seocho.benchmarking import normalize_answer

def correct(answer: str, expected: str) -> bool:
    return normalize_answer(expected) in normalize_answer(answer)

rows = []
for case in cases:
    v = answer_with_vector(case.question)
    g = answer_with_graph(case.question)
    h = answer_hybrid(case.question)
    rows.append({
        "case_id": case.case_id,
        "category": case.category,
        "expected": case.expected_answer,
        "vector_answer": v["answer"],
        "vector_correct": correct(v["answer"], case.expected_answer),
        "vector_total_ms": v["retrieval_ms"] + v["generation_ms"],
        "graph_answer": g["answer"],
        "graph_correct": correct(g["answer"], case.expected_answer),
        "graph_total_ms": g["retrieval_ms"] + g["generation_ms"],
        "hybrid_answer": h["answer"],
        "hybrid_correct": correct(h["answer"], case.expected_answer),
        "hybrid_total_ms": h["retrieval_ms"] + h["generation_ms"],
    })

import pandas as pd
df = pd.DataFrame(rows)
summary = pd.DataFrame({
    "path": ["vector", "graph", "hybrid"],
    "correct_rate": [df["vector_correct"].mean(), df["graph_correct"].mean(), df["hybrid_correct"].mean()],
    "avg_total_ms": [df["vector_total_ms"].mean(), df["graph_total_ms"].mean(), df["hybrid_total_ms"].mean()],
})
print("Per-question detail:")
print(df[["case_id", "category", "vector_correct", "graph_correct", "hybrid_correct"]].to_string(index=False))
print("\nSummary:")
print(summary.to_string(index=False))

## Cleanup

Tutorial artifacts live under `.seocho/finder_tutorial/`. Delete that directory to reset.

## Visualize the constructed graph

Pull the workspace's nodes and relationships back from Neo4j and draw them inline. Open the **Neo4j Browser** at http://localhost:7474 (login `neo4j` / `tutorialspw`) for live exploration:

```cypher
MATCH (n)-[r]->(m) WHERE n._workspace_id = "finder_tutorial" RETURN n,r,m LIMIT 50
```

In [ ]:
from examples.finder.lib.graph_viz import draw_lpg, fetch_lpg_subgraph
import matplotlib.pyplot as plt

viz_nodes, viz_rels = fetch_lpg_subgraph(graph_store, workspace_id=WORKSPACE_ID, limit=80)
print(f"Drawing {len(viz_nodes)} nodes / {len(viz_rels)} relationships from Neo4j")
fig = draw_lpg(viz_nodes, viz_rels, title="FinDER graph extracted via Seocho → Neo4j")
plt.show()

In [ ]:
graph_store.close()
print(f"Artifacts: {WORKDIR}")
print("Neo4j data persists in DozerDB — drop the workspace to reset:")
print(f"  graph_store.execute_write(\"MATCH (n) WHERE n._workspace_id = '{WORKSPACE_ID}' DETACH DELETE n\")")